# Week 6 — Part 01: From scripts to pipelines (stages + artifacts)

**Estimated time:** 75–120 minutes

## What success looks like (end of Part 01)

- You can name the pipeline stages and state each stage’s input/output artifacts.
- You can run a minimal pipeline skeleton end-to-end.
- You can identify a missing artifact immediately (stage contract check fails fast).

### Checkpoint

After running this notebook, you should be able to:

- point to a `pipeline` definition that lists stages in order
- explain what would be saved under `output/` for at least two stages

## Learning Objectives

- Define pipeline stages with clear inputs/outputs
- Understand how artifacts improve reproducibility and debugging
- Design stage contracts (inputs, outputs, invariants)
- Implement a simple pipeline skeleton with explicit stages

**Lab tutorial**: [01_pipeline_design.md](./01_pipeline_design.md) - Detailed walkthrough and learning objectives

### What this part covers
This notebook introduces **pipeline design** — organizing your code into a sequence of stages where each stage has clear inputs, outputs, and a single responsibility.

**Why pipelines instead of one big script?**
- **Debugging:** When something fails, you know exactly which stage failed and can inspect its inputs/outputs
- **Reproducibility:** Intermediate artifacts let you re-run from any checkpoint without redoing expensive work
- **Testability:** Each stage can be tested independently with known inputs and expected outputs

**The project pipeline has 5 stages:** Load → Profile → Compress → LLM → Report

## Overview

A pipeline is a sequence of stages.

Each stage should have:

- clear inputs
- clear outputs
- a single responsibility

This structure improves:

- debugging (you can isolate failures)
- reproducibility (you save intermediate artifacts)

---

## Underlying theory: pipelines make dataflow explicit

You can view a pipeline as a composition of functions:

$$
Y = (f_k \circ f_{k-1} \circ \cdots \circ f_1)(X)
$$

Each stage should have a small, testable contract:

- **inputs**: what files/values it needs
- **outputs**: what artifacts it produces
- **invariants**: what must be true after it runs (schema, counts, non-empty, etc.)

Practical implication:

- a bug is easier to locate because you can bisect stages
- you can cache/reuse artifacts (don’t redo expensive work unnecessarily)
- you can re-run only the stage you changed (faster iteration)

### What this cell does
Defines `Stage` (a named stage with a run function and expected output keys) and `run_pipeline()` — the orchestrator that runs stages in order and validates each stage's outputs.

**Key design: output contract checking**
After each stage runs, `run_pipeline()` checks that every key listed in `stage.outputs` is present in the context dict. If a stage forgets to produce an output, you get an immediate, clear error: `"stage profile did not produce output: profile"` — not a cryptic `KeyError` three stages later.

**The Job Posting Skill Analyzer stages** (the Week 6 capstone theme):
- `load_stage` → reads the real `data/sample_job_postings.csv` into `"loaded"`
- `profile_stage` → adds `"profile"` (row/column counts + top job titles)
- `compress_stage` → adds `"compressed"` (a small summary; never the full CSV)
- `llm_stage` → adds `"llm_raw_response"` and `"llm_output"` (debug stub here)
- `report_stage` → adds `"report"`

**Your task:** Keep the LLM stage as the only stub. In the capstone (`capstone_template/`) you replace `llm_stage` with a real provider call and have each stage write its artifact to `output/` (see the next cell).

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Dict, List

try:
    import pandas as pd
except Exception:  # pragma: no cover
    pd = None


@dataclass
class Stage:
    name: str
    run: Callable[[Dict[str, Any]], Dict[str, Any]]
    outputs: List[str]


def run_pipeline(stages: List[Stage], context: Dict[str, Any]) -> Dict[str, Any]:
    for stage in stages:
        print("running stage:", stage.name)
        context = stage.run(context)
        for key in stage.outputs:
            if key not in context:
                raise RuntimeError("stage %s did not produce output: %s" % (stage.name, key))
    return context


# Minimal Job Posting Skill Analyzer pipeline.
# Load/Profile/Compress/Report are real (if small); the LLM stage is a debug-only
# stub here — the capstone (see capstone_template/) replaces it with a real call.

CSV_PATH = Path("data/sample_job_postings.csv")


def load_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    # Read the real classroom sample when available; otherwise keep the demo runnable.
    ctx["loaded"] = pd.read_csv(CSV_PATH) if (pd is not None and CSV_PATH.exists()) else None
    return ctx


def profile_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    df = ctx["loaded"]
    if df is None:
        ctx["profile"] = {"row_count": 0, "column_count": 0, "top_job_titles": {}}
    else:
        top_titles = (
            {str(k): int(v) for k, v in df["job_title"].value_counts().head(5).to_dict().items()}
            if "job_title" in df.columns
            else {}
        )
        ctx["profile"] = {
            "row_count": int(len(df)),
            "column_count": int(df.shape[1]),
            "top_job_titles": top_titles,
        }
    return ctx


def compress_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    df = ctx["loaded"]
    ctx["compressed"] = {
        "sample_rows": 0 if df is None else min(5, len(df)),
        "note": "send a compressed summary of the job postings, never the full CSV",
    }
    return ctx


def llm_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    # Debug-only placeholder. Final capstone submissions must call a real LLM.
    ctx["llm_raw_response"] = '{"summary": "...", "insights": [], "recommendations": [], "risk_notes": []}'
    ctx["llm_output"] = {"summary": "...", "insights": [], "recommendations": [], "risk_notes": []}
    return ctx


def report_stage(ctx: Dict[str, Any]) -> Dict[str, Any]:
    profile = ctx["profile"]
    ctx["report"] = {
        "title": "Job Posting Skill Analysis",
        "rows": profile["row_count"],
        "top_job_titles": profile["top_job_titles"],
    }
    return ctx


pipeline = [
    Stage("load", load_stage, ["loaded"]),
    Stage("profile", profile_stage, ["profile"]),
    Stage("compress", compress_stage, ["compressed"]),
    Stage("llm", llm_stage, ["llm_raw_response", "llm_output"]),
    Stage("report", report_stage, ["report"]),
]

final_ctx = run_pipeline(pipeline, {})
print("final keys:", sorted(final_ctx.keys()))
print("report:", final_ctx["report"])

### What this cell does
Defines `write_artifact()` — a helper that writes a string to a file (creating parent directories if needed) — and `stage_with_artifact()` — a pattern that combines writing an artifact with returning the path in the context dict.

**Why save artifacts at every stage?**
- If the LLM call (stage 4) fails, you still have `profile.json` and `compressed_input.json` from stages 2 and 3
- You can re-run only stage 4 using the saved `compressed_input.json` — no need to re-profile or re-compress
- Cost control: don't re-call expensive LLM stages when only downstream code changed

**Your task:** Implement `validate_stage_outputs()` with schema checks — not just "key exists" but "key has the expected structure" (e.g., `profile` dict has `row_count` and `column_count` fields).

## Suggested project stages

Make the contract explicit for each stage:

1. **Load**
   - Inputs: `data/*.csv`
   - Outputs: loaded table in memory
   - Pitfalls: dtype drift, missing columns, encoding issues

2. **Profile**
   - Outputs: `output/profile.json`

3. **Compress**
   - Outputs: `output/compressed_input.json`
   - Goal: fit key info into the context window

4. **LLM**
   - Outputs: `output/llm_prompt.txt` and `output/llm_raw_response.txt`

5. **Report**
   - Outputs: `output/report.json` and `output/report.md`

Rule of thumb: if a stage fails, you should still have the previous stage’s artifact saved.

In [ ]:
from pathlib import Path


def write_artifact(path: Path, data: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(data, encoding="utf-8")


def stage_with_artifact(name: str, out_path: Path, payload: str) -> dict:
    write_artifact(out_path, payload)
    return {name: str(out_path)}


ctx = {}
ctx.update(stage_with_artifact("profile", Path("output/profile.json"), "{}"))
ctx.update(stage_with_artifact("compressed", Path("output/compressed_input.json"), "{}"))
print(ctx)

In [ ]:
from typing import Any, Dict, List


def validate_stage_outputs(ctx: Dict[str, Any], required: List[str]) -> None:
    missing = [k for k in required if k not in ctx]
    if missing:
        raise ValueError("missing outputs: %s" % missing)


# TODO: add schema checks (e.g., profile has expected keys)
print("Implement schema checks in validate_stage_outputs().")

## Self-check

- If the LLM call fails, do you still have the profiling artifact?
- Can you re-run only the LLM stage with the saved compressed input?

## References

- Twelve-Factor logs/config mindset: https://12factor.net/